## Improvement of the baseline model  

Iz dodatnog izvora imamo 1633 csv weather file-ova, koji su organizovani po stanicama gde je merena temperatura   
Svaki naziv ovih file-ova je station_id
air_store_info_with_nearest_active_station.csv je skup podataka koji za svaki restoran sadrzi njegovu najblizu stanicu u koloni station_id  
Iz ovog skupa podataka dobijamo vezu restoran - station_id  
Ucitamo samo one koji su nam potrebni, uzimamo calendar_date, avg_temperature i precipitation  
I sve spajamo u jedan veliki dataset

In [1]:
import pandas as pd
import os

original_dataset = pd.read_csv("preprocessed_dataset.csv")
store_station = pd.read_csv("air_store_info_with_nearest_active_station.csv")

In [2]:
print(original_dataset.columns)
print(store_station.columns)

Index(['air_store_id', 'visit_date', 'visitors', 'month', 'reserve_count',
       'reserve_visitors_sum', 'reserve_lead_days_mean', 'afternoon',
       'evening', 'morning', 'has_reservation', 'air_genre_name',
       'air_area_name', 'latitude', 'longitude', 'day_of_week', 'holiday_flg',
       'is_weekend'],
      dtype='object')
Index(['air_store_id', 'air_genre_name', 'air_area_name', 'latitude',
       'longitude', 'latitude_str', 'longitude_str', 'station_id',
       'station_latitude', 'station_longitude', 'station_vincenty',
       'station_great_circle'],
      dtype='object')


Potrebne su nam kolone: 'air_store_id', 'visit_date', 'visitors' iz jednog  
i 'air_store_id' 'station_id' iz drugog dataseta

In [3]:
store_station = store_station[["air_store_id", "station_id"]].copy()

Spajamo po stanicama

In [4]:
original_dataset_with_stations = original_dataset.merge(
    store_station,
    on="air_store_id",
    how="left"
)

Svakom restoranu smo pridruzili njegovu najblizu stanicu

In [5]:
print(original_dataset_with_stations["station_id"].isna().sum())

0


Proverili smo da li ima restorana bez stanice

In [6]:
station_ids = original_dataset_with_stations["station_id"].dropna().unique()
print(station_ids)

['tokyo__tokyo-kana__tonokyo' 'tokyo__nerima-kana__nerima'
 'tokyo__setagaya-kana__setagaya'
 'tokyo__edogawa-seaside-kana__edgawawinkai'
 'saitama__tokorozawa-kana__tokorozawa' 'tokyo__fuchu-kana__fuku'
 'tokyo__haneda-kana__haneda'
 'kanagawa__sagamihara-central-kana__sagamiharasuki'
 'kanagawa__hiyoshi-kana__hiyoshi']


Ovako smo izvukli samo one stanice koje su vezane za nase restorane, to je 9 weather file-ova

In [7]:
import os

weather_folder = r"C:\Users\user\Downloads\archive\1-1-16_5-31-17_Weather\1-1-16_5-31-17_Weather"

weather_dfs = []

for station_id in station_ids:
    file_path = os.path.join(weather_folder, f"{station_id}.csv")
    
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        
        df = df[["calendar_date", "avg_temperature", "precipitation"]].copy()
        df["station_id"] = station_id
        
        weather_dfs.append(df)
    else:
        print(f"Fajl ne postoji: {file_path}")

Spajanje svih df u jedan

In [8]:
weather = pd.concat(weather_dfs, ignore_index=True)

In [9]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4653 entries, 0 to 4652
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   calendar_date    4653 non-null   object 
 1   avg_temperature  3102 non-null   float64
 2   precipitation    4404 non-null   float64
 3   station_id       4653 non-null   object 
dtypes: float64(2), object(2)
memory usage: 145.5+ KB


In [10]:
original_dataset_with_stations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133063 entries, 0 to 133062
Data columns (total 19 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   air_store_id            133063 non-null  object 
 1   visit_date              133063 non-null  object 
 2   visitors                133063 non-null  int64  
 3   month                   133063 non-null  int64  
 4   reserve_count           133063 non-null  float64
 5   reserve_visitors_sum    133063 non-null  float64
 6   reserve_lead_days_mean  133063 non-null  float64
 7   afternoon               133063 non-null  float64
 8   evening                 133063 non-null  float64
 9   morning                 133063 non-null  float64
 10  has_reservation         133063 non-null  int64  
 11  air_genre_name          133063 non-null  object 
 12  air_area_name           133063 non-null  object 
 13  latitude                133063 non-null  float64
 14  longitude           

Pretvaramo datume svuda u isti tip

In [11]:
original_dataset_with_stations["visit_date"] = pd.to_datetime(original_dataset_with_stations["visit_date"])
weather["calendar_date"] = pd.to_datetime(weather["calendar_date"])

Konacno spajanje originalnog skupa podataka sa podacima o vremenskim prilikama

In [12]:
final = original_dataset_with_stations.merge(
    weather,
    left_on=["station_id", "visit_date"],
    right_on=["station_id", "calendar_date"],
    how="left"
)

In [13]:
final.head()

,air_store_id,visit_date,visitors,month,reserve_count,reserve_visitors_sum,reserve_lead_days_mean,afternoon,evening,morning,...,air_area_name,latitude,longitude,day_of_week,holiday_flg,is_weekend,station_id,calendar_date,avg_temperature,precipitation
0,air_ba937bf13d40fb24,2016-01-13,25,1,0.0,0.0,0.0,0.0,0.0,0.0,...,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,Wednesday,0,0,tokyo__tokyo-kana__tonokyo,2016-01-13,4.3,NaN
1,air_ba937bf13d40fb24,2016-01-14,32,1,0.0,0.0,0.0,0.0,0.0,0.0,...,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,Thursday,0,0,tokyo__tokyo-kana__tonokyo,2016-01-14,6.0,NaN
2,air_ba937bf13d40fb24,2016-01-15,29,1,0.0,0.0,0.0,0.0,0.0,0.0,...,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,Friday,0,1,tokyo__tokyo-kana__tonokyo,2016-01-15,5.6,0.0
3,air_ba937bf13d40fb24,2016-01-16,22,1,0.0,0.0,0.0,0.0,0.0,0.0,...,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,Saturday,0,1,tokyo__tokyo-kana__tonokyo,2016-01-16,6.5,NaN
4,air_ba937bf13d40fb24,2016-01-18,6,1,0.0,0.0,0.0,0.0,0.0,0.0,...,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,Monday,0,0,tokyo__tokyo-kana__tonokyo,2016-01-18,2.8,67.0


Izbacujemo calendar_date, ovde necemo izbaciti station_id jer nam koristi kasnije za imputaciju

In [14]:
final.drop(columns=['calendar_date'], inplace = True)

In [15]:
final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133063 entries, 0 to 133062
Data columns (total 21 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   air_store_id            133063 non-null  object        
 1   visit_date              133063 non-null  datetime64[ns]
 2   visitors                133063 non-null  int64         
 3   month                   133063 non-null  int64         
 4   reserve_count           133063 non-null  float64       
 5   reserve_visitors_sum    133063 non-null  float64       
 6   reserve_lead_days_mean  133063 non-null  float64       
 7   afternoon               133063 non-null  float64       
 8   evening                 133063 non-null  float64       
 9   morning                 133063 non-null  float64       
 10  has_reservation         133063 non-null  int64         
 11  air_genre_name          133063 non-null  object        
 12  air_area_name           133063

---------------
Kreiramo jos nekoliko dodatnih atributa i sortiramo po datumima zbog vremenskih serija

In [16]:
final = final.sort_values(["air_store_id", "visit_date"]).reset_index(drop=True)

In [17]:
print(final['air_area_name'].nunique())
print(final['air_genre_name'].nunique())
print(final['day_of_week'].nunique())

51
13
7


Za prvu kategoriju cemo koristi Target encoder a za ostale OneHot

Pravimo atribute koji se odnose na par dana ili dan pre kako bismo uocili moguce obrasce u broju gostiju u nekom blizem periodu

In [18]:
final["visitors_lag_1"] = final.groupby("air_store_id")["visitors"].shift(1)
final["visitors_lag_7"] = final.groupby("air_store_id")["visitors"].shift(7)
final["visitors_lag_28"] = final.groupby("air_store_id")["visitors"].shift(28)

Koristimo rolling kako bismo uhvatile statistiki za odredjeni period, takodje po broju gostiju

In [19]:
final["visitors_roll_mean_7"] = (
    final.groupby("air_store_id")["visitors"]
      .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
)
final["visitors_roll_mean_28"] = (
    final.groupby("air_store_id")["visitors"]
      .transform(lambda x: x.shift(1).rolling(28, min_periods=1).mean())
)
final["visitors_roll_std_7"] = (
    final.groupby("air_store_id")["visitors"]
      .transform(lambda x: x.shift(1).rolling(7, min_periods=2).std())
)

Gledamo ponasanje istih dana u nedelji, expanding mean racuna kumulativni prosek nad shiftovanim vrednostima

In [20]:
final["dow_mean_past"] = (
    final.groupby(["air_store_id", "day_of_week"])["visitors"]
      .transform(lambda x: x.shift(1).expanding().mean())
)

Prosecan broj ljudi po rezervaciji

In [21]:
final["reserve_visitors_per_booking"] = (
    final["reserve_visitors_sum"] / final["reserve_count"].replace(0, pd.NA)
)
final["reserve_visitors_per_booking"] = final["reserve_visitors_per_booking"].fillna(0)

C:\Users\user\AppData\Local\Temp\ipykernel_10620\743599926.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final["reserve_visitors_per_booking"] = final["reserve_visitors_per_booking"].fillna(0)


Rolling za temperaturu u prethodna tri dana

In [22]:
final["temp_rolling_3"] = (

    final.groupby("air_area_name")["avg_temperature"]
      .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

In [23]:
final.head()

,air_store_id,visit_date,visitors,month,reserve_count,reserve_visitors_sum,reserve_lead_days_mean,afternoon,evening,morning,...,precipitation,visitors_lag_1,visitors_lag_7,visitors_lag_28,visitors_roll_mean_7,visitors_roll_mean_28,visitors_roll_std_7,dow_mean_past,reserve_visitors_per_booking,temp_rolling_3
0,air_00a91d42b08b08d9,2016-07-01,35,7,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
1,air_00a91d42b08b08d9,2016-07-02,9,7,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,35.0,NaN,NaN,35.000000,35.000000,NaN,NaN,0.0,25.6
2,air_00a91d42b08b08d9,2016-07-04,20,7,0.0,0.0,0.0,0.0,0.0,0.0,...,1.5,9.0,NaN,NaN,22.000000,22.000000,18.384776,NaN,0.0,26.3
3,air_00a91d42b08b08d9,2016-07-05,25,7,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,20.0,NaN,NaN,21.333333,21.333333,13.051181,NaN,0.0,26.8
4,air_00a91d42b08b08d9,2016-07-06,29,7,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,25.0,NaN,NaN,22.250000,22.250000,10.812801,NaN,0.0,25.5


In [24]:
final.shape

(133063, 30)

In [25]:
final.isna().sum()

air_store_id                        0
visit_date                          0
visitors                            0
month                               0
reserve_count                       0
reserve_visitors_sum                0
reserve_lead_days_mean              0
afternoon                           0
evening                             0
morning                             0
has_reservation                     0
air_genre_name                      0
air_area_name                       0
latitude                            0
longitude                           0
day_of_week                         0
holiday_flg                         0
is_weekend                          0
station_id                          0
avg_temperature                 20878
precipitation                   43211
visitors_lag_1                    444
visitors_lag_7                   3108
visitors_lag_28                 12424
visitors_roll_mean_7              444
visitors_roll_mean_28             444
visitors_rol

In [26]:
final.to_csv("final.csv", index=False)

--------  
Modelujemo sa boljim podacima

In [27]:
final = final.sort_values(by=["air_store_id", "visit_date"])

In [28]:
train = final[final["visit_date"] < "2017-04-01"].copy()
test = final[final["visit_date"] >= "2017-04-01"].copy()

In [29]:
median_per_restaurant_train = (
    train.groupby("air_store_id")["visitors"]
    .median()
    .reset_index(name="median_visitors")
)

In [30]:
train = train.merge(median_per_restaurant_train, on="air_store_id", how="left")
test = test.merge(median_per_restaurant_train, on="air_store_id", how="left")

In [31]:
train["target"] = (train["visitors"] > train["median_visitors"]).astype(int)
test["target"] = (test["visitors"] > test["median_visitors"]).astype(int)

In [32]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124619 entries, 0 to 124618
Data columns (total 32 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   air_store_id                  124619 non-null  object        
 1   visit_date                    124619 non-null  datetime64[ns]
 2   visitors                      124619 non-null  int64         
 3   month                         124619 non-null  int64         
 4   reserve_count                 124619 non-null  float64       
 5   reserve_visitors_sum          124619 non-null  float64       
 6   reserve_lead_days_mean        124619 non-null  float64       
 7   afternoon                     124619 non-null  float64       
 8   evening                       124619 non-null  float64       
 9   morning                       124619 non-null  float64       
 10  has_reservation               124619 non-null  int64         
 11  air_genre_nam

In [33]:
cols_to_drop = ["target", "visitors", "air_store_id", "visit_date", "median_visitors"]

X_train = train.drop(columns=cols_to_drop)
y_train = train["target"]

X_test = test.drop(columns=cols_to_drop)
y_test = test["target"]

In [34]:
y_train.value_counts(normalize=True)

target
0    0.529999
1    0.470001
Name: proportion, dtype: float64

In [35]:
y_test.value_counts(normalize=True)

target
0    0.51658
1    0.48342
Name: proportion, dtype: float64

Da bi pipeline imao smisla, potrebno je napraviti logiku imputacije avg_temperature i precipitation

In [36]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

class WeatherImputer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        station_col="station_id",
        month_col="month",
        temp_col="avg_temperature",
        precip_col="precipitation"
    ):
        self.station_col = station_col
        self.month_col = month_col
        self.temp_col = temp_col
        self.precip_col = precip_col

    def fit(self, X, y=None):
        X = X.copy()

        # za temperaturu, ako imamo i stanicu i mesec
        self.temp_station_month_ = (
            X.groupby([self.station_col, self.month_col])[self.temp_col]
            .median()
            .to_dict()
        ) # za temperaturu ako imamo samo mesec
        self.temp_month_ = (
            X.groupby(self.month_col)[self.temp_col]
            .median()
            .to_dict()
        )
        self.temp_global_ = X[self.temp_col].median()

        # isto i za padavine
        self.prec_station_month_ = (
            X.groupby([self.station_col, self.month_col])[self.precip_col]
            .median()
            .to_dict()
        )
        self.prec_month_ = (
            X.groupby(self.month_col)[self.precip_col]
            .median()
            .to_dict()
        )
        self.prec_global_ = X[self.precip_col].median()

        return self

    def transform(self, X):
        X = X.copy()

        # missing indikatori
        X["temp_missing"] = X[self.temp_col].isna().astype(int)
        X["precipitation_missing"] = X[self.precip_col].isna().astype(int)

        def fill_temp(row):
            if pd.notnull(row[self.temp_col]):
                return row[self.temp_col]

            key = (row[self.station_col], row[self.month_col])

            if key in self.temp_station_month_ and pd.notnull(self.temp_station_month_[key]):
                return self.temp_station_month_[key]

            if row[self.month_col] in self.temp_month_ and pd.notnull(self.temp_month_[row[self.month_col]]):
                return self.temp_month_[row[self.month_col]]

            return self.temp_global_

        def fill_prec(row):
            if pd.notnull(row[self.precip_col]):
                return row[self.precip_col]

            key = (row[self.station_col], row[self.month_col])

            if key in self.prec_station_month_ and pd.notnull(self.prec_station_month_[key]):
                return self.prec_station_month_[key]

            if row[self.month_col] in self.prec_month_ and pd.notnull(self.prec_month_[row[self.month_col]]):
                return self.prec_month_[row[self.month_col]]

            return self.prec_global_

        X[self.temp_col] = X.apply(fill_temp, axis=1)
        X[self.precip_col] = X.apply(fill_prec, axis=1)

        X["is_rain"] = (X[self.precip_col] > 0).astype(int)

        return X

### Logisticka regresija

In [37]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


target_encoded_feature = ["air_area_name"]
onehot_features = ["day_of_week", "air_genre_name"]
categorical_features = target_encoded_feature + onehot_features

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features + ["station_id", "avg_temperature", "precipitation"]
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

onehot_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

target_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("target", TargetEncoder(target_type="binary", smooth="auto", cv=5, random_state=42))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("onehot", onehot_transformer, onehot_features),
    ("target", target_transformer, target_encoded_feature)
])

log_reg_pipeline = Pipeline(steps=[
    ("weather_imputer", WeatherImputer(
        station_col="station_id",
        month_col="month",
        temp_col="avg_temperature",
        precip_col="precipitation"
    )),
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42,
        C=0.1,
        penalty="l1",
        solver="liblinear"
    ))
])

log_reg_pipeline.fit(X_train, y_train)
y_pred = log_reg_pipeline.predict(X_test)
y_proba = log_reg_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label=1))
print("Recall:", recall_score(y_test, y_pred, pos_label=1))
print("F1-score:", f1_score(y_test, y_pred, pos_label=1))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Accuracy: 0.6470866887730933
Precision: 0.6672738312082575
Recall: 0.5384615384615384
F1-score: 0.5959869848156182
ROC-AUC: 0.6945426527843581


### Random Forest

In [38]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

target_encoded_feature = ["air_area_name"]
onehot_features = ["day_of_week", "air_genre_name"]
categorical_features = target_encoded_feature + onehot_features

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features + ["station_id", "avg_temperature", "precipitation"]
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

onehot_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

target_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("target", TargetEncoder(target_type="binary", smooth="auto", cv=5, random_state=42))
])

preprocessor_rf = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("onehot", onehot_transformer, onehot_features),
    ("target", target_transformer, target_encoded_feature)
])

rf_pipeline = Pipeline(steps=[
        ("weather_imputer", WeatherImputer(
        station_col="station_id",
        month_col="month",
        temp_col="avg_temperature",
        precip_col="precipitation"
    )),
    ("preprocessor", preprocessor_rf),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])


rf_pipeline.fit(X_train, y_train)

y_pred = rf_pipeline.predict(X_test)
y_proba = rf_pipeline.predict_proba(X_test)[:, 1]


print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label=1))
print("Recall:", recall_score(y_test, y_pred, pos_label=1))
print("F1-score:", f1_score(y_test, y_pred, pos_label=1))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Accuracy: 0.6761013737565135
Precision: 0.7045247494685697
Recall: 0.5683488486036257
F1-score: 0.6291525423728813
ROC-AUC: 0.7283432638701214


### LightGBM

In [39]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from lightgbm import LGBMClassifier

target_encoded_feature = ["air_area_name"]
onehot_features = ["day_of_week", "air_genre_name"]
categorical_features = target_encoded_feature + onehot_features

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features + ["station_id", "avg_temperature", "precipitation"]
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

onehot_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

target_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("target", TargetEncoder(target_type="binary", smooth="auto", cv=5, random_state=42))
])

preprocessor_lgbm = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("onehot", onehot_transformer, onehot_features),
    ("target", target_transformer, target_encoded_feature)
])

lgbm_pipeline = Pipeline(steps=[
        ("weather_imputer", WeatherImputer(
        station_col="station_id",
        month_col="month",
        temp_col="avg_temperature",
        precip_col="precipitation"
    )),
    ("preprocessor", preprocessor_lgbm),
    ("model", LGBMClassifier(
        n_estimators=200,
        learning_rate=0.01,
        num_leaves=31,
        random_state=42
    ))
])


lgbm_pipeline.fit(X_train, y_train)

y_pred = lgbm_pipeline.predict(X_test)
y_proba = lgbm_pipeline.predict_proba(X_test)[:, 1]


print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label=1))
print("Recall:", recall_score(y_test, y_pred, pos_label=1))
print("F1-score:", f1_score(y_test, y_pred, pos_label=1))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

[LightGBM] [Info] Number of positive: 58571, number of negative: 66048
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003317 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2678
[LightGBM] [Info] Number of data points in the train set: 124619, number of used features: 42
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.470001 -> initscore=-0.120142
[LightGBM] [Info] Start training from score -0.120142
Accuracy: 0.6697063003315964
Precision: 0.7053032708796443
Recall: 0.5440960313571779
F1-score: 0.6142995436315862
ROC-AUC: 0.7237092660972755


C:\Users\user\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\user\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### XGBoost

In [40]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

target_encoded_feature = ["air_area_name"]
onehot_features = ["day_of_week", "air_genre_name"]
categorical_features = target_encoded_feature + onehot_features

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features + ["station_id", "avg_temperature", "precipitation"]
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

onehot_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

target_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("target", TargetEncoder(target_type="binary", smooth="auto", cv=5, random_state=42))
])

preprocessor_xgb = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("onehot", onehot_transformer, onehot_features),
    ("target", target_transformer, target_encoded_feature)
])

xgb_pipeline = Pipeline(steps=[
    ("weather_imputer", WeatherImputer(
        station_col="station_id",
        month_col="month",
        temp_col="avg_temperature",
        precip_col="precipitation"
    )),
    ("preprocessor", preprocessor_xgb),
    ("model", XGBClassifier(
        n_estimators=200,
        learning_rate=0.01,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss"
    ))
])


xgb_pipeline.fit(X_train, y_train)

y_pred = xgb_pipeline.predict(X_test)
y_proba = xgb_pipeline.predict_proba(X_test)[:, 1]


print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label=1))
print("Recall:", recall_score(y_test, y_pred, pos_label=1))
print("F1-score:", f1_score(y_test, y_pred, pos_label=1))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Accuracy: 0.670772145902416
Precision: 0.7108160621761658
Recall: 0.5377266046055855
F1-score: 0.6122733612273361
ROC-AUC: 0.7257780717662966


Vrsimo optimizaciju hiperparametara modela Random Forest zato sto je dao najvisu AUC vrednost

In [41]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier

tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [5, 10],
    "model__min_samples_leaf": [5, 10, 20],
    "model__max_leaf_nodes": [20, 50]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=tscv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Najbolji parametri:", grid_search.best_params_)
print("Najbolji CV ROC-AUC:", grid_search.best_score_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Najbolji parametri: {'model__max_depth': 10, 'model__max_leaf_nodes': 50, 'model__min_samples_leaf': 10, 'model__n_estimators': 300}
Najbolji CV ROC-AUC: 0.6835404202459234
